**Week 03 - Python Optimization Assignment <br>
Daipayan Bera <br>
02-09-2025**

In [251]:
#Importing libraries
import pandas as pd
import numpy as np
from math import *

In [252]:
#Importing the clinic dataset
clinics_df = pd.read_csv('clinics.csv', sep='|')

In [253]:
#EDA
clinics_df.head()

,bizID,bizCat,bizCatSub,bizName,bizAddr,bizCity,bizState,bizZip,bizPhone,bizFax,...,bizURL,locAreaCode,locFIPS,locTimeZone,locDST,locLat,locLong,locMSA,locPMSA,locCounty
0,1,Clinics,Clinics,Hino Ronald H MD,98-151 Pali Momi Street Suite 142,Aiea,HI,96701,(808)487-2477,NaN,...,NaN,808,15003,PST-2,N,21.3980,-157.8981,3320.0,NaN,Honolulu
1,2,Clinics,Clinics,Farmer Joesph F Md,1225 Breckenridge Drive,Little Rock,AR,72205,(501)225-2594,NaN,...,NaN,501,5119,CST,Y,34.7495,-92.3533,4400.0,NaN,Pulaski
2,3,Clinics,Clinics & Medical Centers,Najjar Fadi Md,1155 West Linda Avenue Suite B,Hermiston,OR,97838,(541)289-1122,NaN,...,NaN,541,41059,PST,Y,45.8456,-119.2817,NaN,NaN,Umatilla
3,4,Clinics,Clinics & Medical Centers,Kittson Memorial Upper Level Nursing Home,1010 South Birch Avenue,Hallock,MN,56728,(218)843-2525,NaN,...,NaN,218,27069,CST,Y,48.7954,-97.0090,NaN,NaN,Kittson
4,5,Clinics,Clinics & Medical Centers,Thompson Robert B Md,100 North Eagle Creek Drive,Lexington,KY,40509,(859)258-4000,NaN,...,www.lexingtonclinic.com,859,21067,EST,Y,37.9935,-84.3712,4280.0,NaN,Fayette


In [254]:
#EDA
clinics_df.describe()

,bizID,bizZip,bizFax,bizEmail,locAreaCode,locFIPS,locLat,locLong,locMSA,locPMSA
count,30.000000,30.000000,0.0,0.0,30.000000,30.000000,30.000000,30.000000,13.000000,6.000000
mean,15.500000,52672.566667,NaN,NaN,597.633333,21606.866667,37.860970,-93.699020,4182.307692,4553.333333
std,8.803408,28277.921265,NaN,NaN,213.911436,16341.733474,5.760449,17.829063,2442.615244,3080.303015
min,1.000000,1201.000000,NaN,NaN,205.000000,1073.000000,21.398000,-157.898100,520.000000,720.000000
25%,8.250000,31730.500000,NaN,NaN,479.250000,6567.500000,34.034600,-96.953425,2670.000000,2320.000000
50%,15.500000,55607.500000,NaN,NaN,601.000000,18082.000000,38.508650,-89.870800,3560.000000,4620.000000
75%,22.750000,72578.500000,NaN,NaN,756.250000,28902.500000,41.773125,-84.147550,6200.000000,6380.000000
max,30.000000,97838.000000,NaN,NaN,970.000000,55069.000000,48.795400,-71.440500,9000.000000,8840.000000


In [255]:
def haversine(lat1, lon1, lat2, lon2):
    '''
    This is the function based on haversine formula which determines the great-circle distance between two points on earth given their longitudes and latitudes.
    Source : https://medium.com/upside-engineering/a-beginners-guide-to-optimizing-pandas-code-for-speed-c09ef2c6a4d6  
    
    Doctests:
    
    >>> haversine(38.6227, -90.2382, 21.398, -157.8981)
    4126.5068622208855
    
    >>> 
    '''
    miles_constant = 3959
    lat1, lon1, lat2, lon2 = map(np.deg2rad, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1 
    dlon = lon2 - lon1 
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a)) 
    mi = miles_constant * c
    return mi

**Running doctest**

In [256]:
import doctest
doctest.run_docstring_examples(haversine, globals(), verbose=True)

Finding tests in NoName
Trying:
    haversine(38.6227, -90.2382, 21.398, -157.8981)
Expecting:
    4126.5068622208855
ok


For tablular result, here I am defining a dataframe with predetermined column names.

In [257]:
result = pd.DataFrame(columns=["Method", "Time"]) #Method will store the different variants of operation and Time column will store the "timeit" result of those operations.

**For loop (We will be passing each row to the loop and call the function)**

In [258]:
timeit_result = %timeit -o clinics_df["Distance"] = haversine_series = [haversine(38.6227, -90.2382, row["locLat"], row["locLong"]) for _, row in clinics_df.iterrows()]
result.loc[len(result)] = ["Forloop", timeit_result] #We are appending the result into the result dataframe

6.2 ms ± 269 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


*We are also inspecting the above loop with line profiler*

In [259]:
#Loading line profiler in the environment
%load_ext line_profiler

The line_profiler extension is already loaded. To reload it, use:
  %reload_ext line_profiler


In [260]:
#Here, we are combining above line of codes in a function as line profiler does not work with assignment operation
def forloopfunc():
    haversine_series = []
    for index, row in clinics_df.iterrows():
        haversine_series.append(haversine(38.6227,-90.2382, row["locLat"], row["locLong"]))
    clinics_df["Distance"] = haversine_series
    
%lprun -f forloopfunc forloopfunc()

**Combination of Apply and Lambda function (We will also be passing each row to the lambda function)**

In [261]:
timeit_result = %timeit -o clinics_df["distance"] = clinics_df.apply(lambda row: haversine(38.6227,-90.2382, row["locLat"], row["locLong"]), axis=1)

result.loc[len(result)] = ["Apply lambda", timeit_result]

3.21 ms ± 284 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


*we are also inspecting the above loop with line profiler*

In [262]:
%lprun -f haversine clinics_df["distance"] = clinics_df.apply(lambda row: haversine(38.6227,-90.2382, row["locLat"], row["locLong"]), axis=1)

**Using Vectorized Pandas series operation (We are passing pandas series to the function)**

In [263]:
timeit_result = %timeit -o clinics_df["distance"] = haversine(38.6227,-90.2382, clinics_df["locLat"], clinics_df["locLong"])

result.loc[len(result)] = ["Vectorized pandas series", timeit_result]

3.52 ms ± 279 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


*we are also inspecting the above loop with line profiler*

In [264]:
%lprun -f haversine clinics_df["distance"] = haversine(38.6227,-90.2382, clinics_df["locLat"], clinics_df["locLong"])

**Using Vectorized NumPy array operation (We are passing NumPy arrays to the function)**

In [265]:
timeit_result = %timeit -o clinics_df["distance"] = haversine(38.6227,-90.2382, clinics_df["locLat"].values, clinics_df["locLong"].values)

result.loc[len(result)] = ["Vectorized Numpy arrays", timeit_result]

336 µs ± 11.8 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


*we are also inspecting the above loop with line profiler*

In [266]:
%lprun -f haversine clinics_df["distance"] = haversine(38.6227,-90.2382, clinics_df["locLat"].values, clinics_df["locLong"].values)

In [267]:
#Printing the tabular result
result

,Method,Time
0,Forloop,6.2 ms ± 269 µs per loop (mean ± std. dev. of ...
1,Apply lambda,3.21 ms ± 284 µs per loop (mean ± std. dev. of...
2,Vectorized pandas series,3.52 ms ± 279 µs per loop (mean ± std. dev. of...
3,Vectorized Numpy arrays,336 µs ± 11.8 µs per loop (mean ± std. dev. of...


**Next, we are going to a external python library called Cython, which helps to define python code to C code and make it more time efficient.**

In [269]:
#Loading cython
%load_ext cython 

The cython extension is already loaded. To reload it, use:
  %reload_ext cython


*#This helps to check how much of the following written code was converted to C code and how much remained as Python code*

In [289]:
%%cython -a

# Haversine cythonized (no other edits)
import numpy as np
cpdef haversine_cy(lat1, lon1, lat2, lon2):
    miles_constant = 3959
    lat1, lon1, lat2, lon2 = map(np.deg2rad, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1 
    dlon = lon2 - lon1 
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a)) 
    mi = miles_constant * c
    return mi

**Here, we can see that from white lines (transformed C code) that we have not changed much of the python code (yellow lines). So, this is still not much efficient.**

In [290]:
timeit_result = %timeit -o clinics_df["Distance"] = haversine_series = [haversine_cy(38.6227, -90.2382, row["locLat"], row["locLong"]) for _, row in clinics_df.iterrows()]
result.loc[len(result)] = ["Forloop_Cython", timeit_result] 

5.43 ms ± 582 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [291]:
timeit_result = %timeit -o clinics_df["distance"] = clinics_df.apply(lambda row: haversine_cy(38.6227,-90.2382, row["locLat"], row["locLong"]), axis=1)

result.loc[len(result)] = ["Apply lambda_Cython", timeit_result]

3.75 ms ± 254 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [292]:
timeit_result = %timeit -o clinics_df["distance"] = haversine_cy(38.6227,-90.2382, clinics_df["locLat"], clinics_df["locLong"])

result.loc[len(result)] = ["Vectorized pandas series_Cython", timeit_result]

3.89 ms ± 204 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [293]:
timeit_result = %timeit -o clinics_df["distance"] = haversine_cy(38.6227,-90.2382, clinics_df["locLat"].values, clinics_df["locLong"].values)

result.loc[len(result)] = ["Vectorized Numpy arrays_Cython", timeit_result]

302 µs ± 28.4 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


**We can compare the result of Python and Cython operations and can see that it has barely made any difference.**

In [294]:
result

,Method,Time
0,Forloop,6.2 ms ± 269 µs per loop (mean ± std. dev. of ...
1,Apply lambda,3.21 ms ± 284 µs per loop (mean ± std. dev. of...
2,Vectorized pandas series,3.52 ms ± 279 µs per loop (mean ± std. dev. of...
3,Vectorized Numpy arrays,336 µs ± 11.8 µs per loop (mean ± std. dev. of...
4,Forloop_Cython,5.86 ms ± 214 µs per loop (mean ± std. dev. of...
5,Apply lambda_Cython,3.67 ms ± 142 µs per loop (mean ± std. dev. of...
6,Vectorized pandas series_Cython,3.64 ms ± 170 µs per loop (mean ± std. dev. of...
7,Vectorized Numpy arrays_Cython,344 µs ± 34.9 µs per loop (mean ± std. dev. of...
8,Forloop_Cython_Dtypes,4.36 ms ± 393 µs per loop (mean ± std. dev. of...
9,Apply lambda_Cython_Dtypes,2.63 ms ± 236 µs per loop (mean ± std. dev. of...


**Next, we have used the same code as Sofia Heisler (https://medium.com/upside-engineering/a-beginners-guide-to-optimizing-pandas-code-for-speed-c09ef2c6a4d6), which modifies the python code with C code and C datatypes.**

In [295]:
%%cython -a
# Haversine cythonized
from libc.math cimport sin, cos, acos, asin, sqrt

cdef deg2rad_cy(float deg):
    cdef float rad
    rad = 0.01745329252*deg
    return rad
    
cpdef haversine_cy_dtyped(float lat1, float lon1, float lat2, float lon2):
    cdef: 
        float dlon
        float dlat
        float a
        float c
        float mi
    
    lat1, lon1, lat2, lon2 = map(deg2rad_cy, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1 
    dlon = lon2 - lon1 
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a)) 
    mi = 3959 * c
    return mi

**We can see the changes in the color of the code; there are a lot of white lines and fewer yellow lines, which determines that we are using more C code than Python code. We can hope that now the operations will be more time efficient.**

In [296]:
timeit_result = %timeit -o clinics_df["Distance"] = haversine_series = [haversine_cy_dtyped(38.6227, -90.2382, row["locLat"], row["locLong"]) for _, row in clinics_df.iterrows()]
result.loc[len(result)] = ["Forloop_Cython_Dtypes", timeit_result] 

4.66 ms ± 249 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [297]:
timeit_result = %timeit -o clinics_df["distance"] = clinics_df.apply(lambda row: haversine_cy_dtyped(38.6227,-90.2382, row["locLat"], row["locLong"]), axis=1)

result.loc[len(result)] = ["Apply lambda_Cython_Dtypes", timeit_result]

3.81 ms ± 811 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


**We won't be able to use the "haversine_cy_dtyped" function and pass pandas series there, as we have typecasted the arguments to float; therefore, it won't be able to receive pandas series or numpy arrays for that matter.**

In [298]:
from tabulate import tabulate
print(tabulate(result, headers='keys', tablefmt='fancy_grid'))

╒════╤═════════════════════════════════╤══════════════════════════════════════════════════════════════════════════╕
│    │ Method                          │ Time                                                                     │
╞════╪═════════════════════════════════╪══════════════════════════════════════════════════════════════════════════╡
│  0 │ Forloop                         │ 6.2 ms ± 269 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)    │
├────┼─────────────────────────────────┼──────────────────────────────────────────────────────────────────────────┤
│  1 │ Apply lambda                    │ 3.21 ms ± 284 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)   │
├────┼─────────────────────────────────┼──────────────────────────────────────────────────────────────────────────┤
│  2 │ Vectorized pandas series        │ 3.52 ms ± 279 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)   │
├────┼─────────────────────────────────┼────────────────────────────────